Connect the PicoEMP's GP0 (orange - trigger) to the target's DIO8 pin

Connect the PicoEMP's ground to a ground (black) pin on the arduino board

Connect the PicoEMP's GP1 (blue - RST) to the target's RST pin

In [1]:
import binascii
import serial
import time
import random
import fastecdsa
from fastecdsa.curve import Curve
from fastecdsa.point import Point
import sys
import os
import numpy as np
from tqdm.notebook import tqdm
import serial
import math

target = None

In [2]:
%run "./picoEMP_setup.ipynb"

In [3]:
pico = PicoEMP('/dev/picoEMP')
pico.reset_target()
pico.setup_internal_control()

Connected to PicoEMP!
status
Status:
- Disarmed
- Not charged
- Timeout disabled
- HVP internal

[status] > 



In [4]:
TARGET_SERIAL_PORT = '/dev/ttyUSB0'
BAUD_RATE = 500000
if target:
    target.close()

pico.reset_target()
target = serial.Serial(TARGET_SERIAL_PORT, BAUD_RATE)
# Wait for Arduino Reset (Nano resets on serial open)
time.sleep(2)
target.reset_input_buffer()

## ECDSA Faults

In [ ]:
## Interactive UI 

import time
import serial
import random
import threading
import ipywidgets as widgets
from IPython.display import display
import csv
import os
import ast

# Globals
stop_attack = False
USE_PICO = True

LOG_FILE = "faults_log.csv"
TOTAL_EXECUTIONS_FILE = "faults_total.csv"

PUBKEY_PREFIX = "Pub  Key: "
PRIVKEY_PREFIX = "Priv Key: "
NONCE_PREFIX = "k2: "
SIGNATURE_PREFIX = "Signature: "
q_PREFIX = "q: "

# ==========================================

# Convert the hex string to a bytes object
def parse_str_to_hex(in_str):
    if in_str:
        try:
            pub_bytes = binascii.unhexlify(in_str)
            return pub_bytes
        except binascii.Error:
            raise(ValueError(f"Error: Hex string '{in_str}' is invalid."))
            
def reset_target():
    pico.reset_target()
    print("Timeout reset!")
    time.sleep(0.5)

def ecdsa_init():
    target = None
    read_keys = 0
    timeout_reset = 4.0

    try:
        pico.reset_target()
        time.sleep(0.5)
        
        target = serial.Serial(TARGET_SERIAL_PORT, BAUD_RATE, timeout=2)
        print(f"Connected to {TARGET_SERIAL_PORT} at {BAUD_RATE} baud.")

        time.sleep(0.5)
        start_time = time.time()

        while read_keys < 2:
            if (time.time() - start_time) > timeout_reset:
                raise TimeoutError("Timeout waiting for ECDSA keys")
            
            # If no data is available, sleep briefly and loop again to check timeout
            if target.in_waiting == 0:
                time.sleep(0.005)
                continue
            # ----------------------------------------------------------------
            
            while target.in_waiting > 0:
                line = target.readline().decode('utf-8', errors='ignore').strip()
                if line:
                    if line.startswith(PRIVKEY_PREFIX):
                        priv_hex_str = line[len(PRIVKEY_PREFIX):].strip()
                        priv_hex = parse_str_to_hex(priv_hex_str)
                        priv_int = int.from_bytes(priv_hex, 'big')
                        read_keys += 1
                    elif line.startswith(PUBKEY_PREFIX):
                        pub_hex_str = line[len(PUBKEY_PREFIX):].strip()
                        pub_hex = parse_str_to_hex(pub_hex_str)
                        pub_int = int.from_bytes(pub_hex, 'big')
                        read_keys += 1

        print("ECDSA initialized")
        return target, priv_int, pub_int, priv_hex, pub_hex

    except serial.SerialException as e:
        print(f"Error: {e}")
        if target and target.is_open:
            target.close()
            print("Connection closed.")
    except KeyboardInterrupt:
        print("\nLoop stopped by user.")
        if target and target.is_open:
            target.close()
            print("Connection closed.")

# ==========================================

def run_attack_ecdsa_ui():

    # Delay Controls & Step Input
    delay_slider = widgets.IntSlider(value=10, min=2, max=3000, step=1, description='Delay:', layout={'width': '300px'})
    delay_minus = widgets.Button(description='-', layout={'width': '40px'}, tooltip='Decrease Delay')
    delay_plus = widgets.Button(description='+', layout={'width': '40px'}, tooltip='Increase Delay')
    delay_step_input = widgets.IntText(value=10, description='Step:', layout={'width': '140px'})
    
    # Pulse Controls & Step Input
    pulse_slider = widgets.IntSlider(value=10, min=5, max=40, step=1, description='Pulse:', layout={'width': '300px'})
    pulse_minus = widgets.Button(description='-', layout={'width': '40px'}, tooltip='Decrease Pulse')
    pulse_plus = widgets.Button(description='+', layout={'width': '40px'}, tooltip='Increase Pulse')
    pulse_step_input = widgets.IntText(value=1, description='Step:', layout={'width': '140px'})

    # Group the sliders, buttons, and step inputs horizontally
    delay_box = widgets.HBox([delay_minus, delay_slider, delay_plus, delay_step_input])
    pulse_box = widgets.HBox([pulse_minus, pulse_slider, pulse_plus, pulse_step_input])

    # SCAN CONTROLS
    enable_scan_checkbox = widgets.Checkbox(value=False, description='Enable Delay Scan', layout={'width': '150px'})
    start_delay_scan = widgets.IntText(value=100, description='Start:', layout={'width': '150px'})
    stop_delay_scan = widgets.IntText(value=1000, description='Stop:', layout={'width': '150px'})
    step_delay_scan = widgets.IntText(value=5, description='Step:', layout={'width': '150px'})
    repetitions_input = widgets.IntText(value=10, description='Repetitions:', layout={'width': '150px'}) 
    
    # TARGETED SCAN CONTROLS
    enable_targeted_checkbox = widgets.Checkbox(value=False, description='Scan Config', layout={'width': '150px'})
    targeted_input = widgets.Text(value='', placeholder='Paste [(delay, pulse), ...] here', description='Configs:', layout={'width': '458px'})

    # UI Layout construction
    scan_row_1 = widgets.HBox([enable_scan_checkbox, start_delay_scan, stop_delay_scan])
    scan_row_2 = widgets.HBox([widgets.HTML("<div style='width: 150px;'></div>"), step_delay_scan, repetitions_input])
    scan_row_3 = widgets.HBox([enable_targeted_checkbox, targeted_input]) 
    scan_box = widgets.VBox([scan_row_1, scan_row_2, scan_row_3])

    n_tests_input = widgets.IntText(value=100, description='N Tests (Manual):', layout={'width': '200px'})
    start_button = widgets.Button(description="Start Attack", button_style='success')
    stop_button = widgets.Button(description="Stop", button_style='danger')
    reset_button = widgets.Button(description="Reset Target", button_style='warning', tooltip='Force Hardware Reset')
    
    # progress bar
    progress_bar = widgets.IntProgress(value=0, min=0, max=1000, description='Progress:', bar_style='info', layout={'width': '400px'})
    
    # HTML Output box for top-down printing
    output_log = widgets.HTML(
        value="", 
        layout={
            'border': '1px solid #ccc', 'height': '400px', 'overflow_y': 'auto', 
            'padding': '10px', 'background-color': '#1e1e1e', 
            'color': '#d4d4d4', 'font-family': 'monospace'
        }
    )
    log_history = []
    
    def log(msg):
        html_msg = str(msg).replace('\n', '<br>')
        log_history.insert(0, html_msg)
        if len(log_history) > 20:
            log_history.pop()
        output_log.value = '<br>'.join(log_history)   

    # UI Layout
    controls = widgets.VBox([
        delay_box, 
        pulse_box,
        scan_box, 
        widgets.HBox([n_tests_input, start_button, stop_button, reset_button]), 
        progress_bar
    ])
    display(controls, output_log)

    # ==========================================
    # Button Callbacks (+/- Logic)
    # ==========================================
    
    def decrease_delay(b): delay_slider.value = max(delay_slider.min, delay_slider.value - delay_step_input.value)
    def increase_delay(b): delay_slider.value = min(delay_slider.max, delay_slider.value + delay_step_input.value)
    def decrease_pulse(b): pulse_slider.value = max(pulse_slider.min, pulse_slider.value - pulse_step_input.value)
    def increase_pulse(b): pulse_slider.value = min(pulse_slider.max, pulse_slider.value + pulse_step_input.value)

    def on_reset_clicked(b):
        html_msg = "<span style='color: #f39c12; font-weight: bold;'>[MANUAL] Hardware reset triggered by user!</span>"
        log_history.insert(0, html_msg)
        if len(log_history) > 50: log_history.pop()
        output_log.value = '<br>'.join(log_history)
        reset_target()

    delay_minus.on_click(decrease_delay)
    delay_plus.on_click(increase_delay)
    pulse_minus.on_click(decrease_pulse)
    pulse_plus.on_click(increase_pulse)
    reset_button.on_click(on_reset_clicked)
    
    # ==========================================
    # Attack Loop thread
    # ==========================================
    
    def attack_task(configs_to_test):

        execution_counts = {}
        
        expected_priv = parse_str_to_hex('1F1E1D1C1B1A191817161514131211100F0E0D0C0B0A09080706050403020100')
        expected_pub = parse_str_to_hex('984225585D2285C138033D6140E3CEF8B91859704E53C313F8B636BA4F9676499734144F46FD19A767A545287C4396B97B69DD38FAAEA8981ADC1A4FED9B401E00')
        expected_signature = parse_str_to_hex('984225585D2285C138033D6140E3CEF8B91859704E53C313F8B636BA4F9676494903EE7B32E5A341E10859AD7E8F3B8ACC149229264C7F0A5322BD83A9616C5E')
        expected_q = parse_str_to_hex('512563FCC2CAB9F3849E17A7ADFAE6BCFFFFFFFFFFFFFFFF00000000FFFFFFFF')
        curve = fastecdsa.curve.P256
        
        expected_q_int = int.from_bytes(expected_q, 'big')
        expected_signature_int = int.from_bytes(expected_signature, 'big')
        RLEN = math.ceil((len(hex(curve.p)) - 2) / 2)
        HASHLEN = math.ceil((len(hex(curve.p)) - 2) / 2)
        G = Point(curve.gx, curve.gy, curve=curve)
        
        item_strings = ['s', 'q']

        log(f'HASH LEN = {HASHLEN}')
        static_data_to_send = bytes([x for x in range(HASHLEN)])
        static_m_hash = int.from_bytes(static_data_to_send, 'big')

        global stop_attack
        target = None
        timeout_reset = 4.0

        if USE_PICO:
            pico.arm()
        log("Initializing ECDSA target. Pico Armed...\n")
            
        try:
            target, priv_int, pub_int, priv_hex, pub_hex = ecdsa_init()
            Q = priv_int * G
            log("ECDSA Init success")
        except TimeoutError:
            log("<span style='color: #e74c3c;'>Fatal Error: ecdsa_init() timed out on first boot.</span>")
            start_button.disabled = False
            return
        
        if target is None:
            log("<span style='color: #e74c3c;'>Fatal Error: ecdsa_init() failed to connect.</span>")
            start_button.disabled = False
            return

        try:
            for i, current_config in enumerate(configs_to_test):
                if stop_attack:
                    log("\n<span style='color: #e74c3c;'>Attack stopped by user.</span>\n")
                    break
                
                try:
                    # update progress bar
                    if i % 5 == 0 or i == len(configs_to_test) - 1:
                        progress_bar.value = i + 1
                    
                    if current_config is not None:
                        delay_cycles, pulse_cycles = current_config
                        
                        # UI OPTIMIZATION
                        if delay_cycles != delay_slider.value:
                            if delay_cycles > delay_slider.max: delay_slider.max = delay_cycles
                            if delay_cycles < delay_slider.min: delay_slider.min = delay_cycles
                            delay_slider.value = delay_cycles 
                            
                        if pulse_cycles != pulse_slider.value:
                            if pulse_cycles > pulse_slider.max: pulse_slider.max = pulse_cycles
                            if pulse_cycles < pulse_slider.min: pulse_slider.min = pulse_cycles
                            pulse_slider.value = pulse_cycles
                    else:
                        delay_cycles = delay_slider.value
                        pulse_cycles = pulse_slider.value

                    # Track total executions
                    config_key = (delay_cycles, pulse_cycles)
                    execution_counts[config_key] = execution_counts.get(config_key, 0) + 1


                    if USE_PICO:
                        pico.configure_fast_trigger(delay_cycles, pulse_cycles)
                        pico.set_fast_trigger()

                    target.write(static_data_to_send)

                    start_time = time.time()
                    signature_received = False
                    
                    while signature_received == False:
                        if stop_attack: break
                        
                        if (time.time() - start_time) > timeout_reset:
                            raise TimeoutError()
                        
                        if target.in_waiting == 0:
                            time.sleep(0.005) 
                            continue
                        
                        while target.in_waiting > 0:
                            line = target.readline().decode('utf-8', errors='ignore').strip()
                            if line:
                                if line.startswith(NONCE_PREFIX):
                                    k_hex = parse_str_to_hex(line[len(NONCE_PREFIX):].strip())
                                    k_int = int.from_bytes(k_hex, 'big')
                                elif line.startswith(q_PREFIX):
                                    q_hex = parse_str_to_hex(line[len(q_PREFIX):].strip())
                                    q_int = int.from_bytes(q_hex, 'big')
                                elif line.startswith(SIGNATURE_PREFIX):
                                    signature_hex = parse_str_to_hex(line[len(SIGNATURE_PREFIX):].strip())
                                    signature_int = int.from_bytes(signature_hex, 'big')
                                    signature_received = True

                    if (time.time() - start_time) > timeout_reset:
                        raise TimeoutError()

                    if q_int == expected_q_int:
                        pass # Verified normally
                    else:
                        # Fault occurred! 
                        log("==== FAULT INJECTED! ====\n")
                        log(f"Params: delay: {delay_cycles} - pulse: {pulse_cycles}\n")

                        expected_items = [expected_signature, expected_q]
                        input_items = [signature_hex, q_hex]      

                        if q_hex != expected_q:
                            faults_to_log = []
                            for byte_idx, (exp_b, act_b) in enumerate(zip(expected_q, q_hex)):
                                if exp_b != act_b:
                                    faults_to_log.append({
                                        'delay': delay_cycles,
                                        'pulse': pulse_cycles,
                                        'byte_index': byte_idx,
                                        'expected_val': f"{exp_b:02X}",
                                        'faulty_val': f"{act_b:02X}"
                                    })
                            
                            if faults_to_log:
                                file_exists = os.path.isfile(LOG_FILE)
                                with open(LOG_FILE, 'a', newline='') as f:
                                    fieldnames = ['delay', 'pulse', 'byte_index', 'expected_val', 'faulty_val']
                                    writer = csv.DictWriter(f, fieldnames=fieldnames)
                                    if not file_exists:
                                        writer.writeheader()
                                    writer.writerows(faults_to_log)
                                    f.flush()
                                    os.fsync(f.fileno())

                        for item in range(len(item_strings)):
                            item_equal = True
                            to_log = "Received {}: ".format(item_strings[item])
                            
                            for x_i, y_i in zip(expected_items[item], input_items[item]):
                                if x_i != y_i:
                                    to_log += "<span style='color: #ff5555; font-weight: bold;'>{:02X}</span>".format(y_i)
                                else:
                                    to_log += "{:02X}".format(y_i)
                            
                            log(to_log)
                
                except TimeoutError:
                    log(f"Target timeout at loop {i}. Resetting...\n")
                    reset_target()
                    target, priv_int, pub_int, priv_hex, pub_hex = ecdsa_init()
                    if target is None:
                        log("<span style='color: #e74c3c;'>Failed to reconnect. Aborting scan.</span>")
                        break
                    continue 

                except Exception as inner_e:
                    log(f"<span style='color: #e67e22;'>EMI/Serial Crash at loop {i}: {inner_e}</span>")
                    
                    try:
                        if target and getattr(target, 'is_open', False):
                            target.close()
                    except Exception: 
                        pass
                    
                    # Give the OS time to physically re-mount the crashed USB devices
                    log("Waiting 3.5 seconds for USB bus OS recovery...")
                    time.sleep(3.5)
                    
                    # Recover the PicoEMP safely
                    try: 
                        pico.disarm() 
                        time.sleep(0.5)
                    except Exception: 
                        pass
                    
                    try:
                        if USE_PICO:
                            pico.arm()
                        reset_target()
                    except Exception as pico_e:
                        log(f"Warning: PicoEMP serial also crashed ({pico_e}). Waiting 2s...")
                        time.sleep(2.0)
                    
                    # Safely catch ecdsa_init failure
                    init_result = ecdsa_init()
                    
                    if init_result is None:
                        log("<span style='color: #e74c3c;'>Failed to reconnect after crash. Aborting scan.</span>")
                        break
                        
                    # Unpack and resume if successful
                    target, priv_int, pub_int, priv_hex, pub_hex = init_result
                    log("<span style='color: #2ecc71;'>Successfully recovered USB connection! Resuming...</span>")
                    continue

        except Exception as outer_e:
            log(f"Fatal Thread Error: {outer_e}")
        finally:
            try:
                if target and getattr(target, 'is_open', False):
                    target.close()
            except Exception: pass
            
            try:
                pico.disarm()
            except Exception: pass
            
            # Write total number of executions to log file
            if execution_counts:
                file_exists = os.path.isfile(TOTAL_EXECUTIONS_FILE)
                try:
                    with open(TOTAL_EXECUTIONS_FILE, 'a', newline='') as f:
                        writer = csv.writer(f)
                        if not file_exists:
                            writer.writerow(['delay', 'pulse', 'total_executions'])
                        for (delay, pulse), count in execution_counts.items():
                            writer.writerow([delay, pulse, count])
                    log("Execution totals saved to disk.")
                except Exception as e:
                    log(f"<span style='color: #e74c3c;'>Failed to save execution totals: {e}</span>")
                    
            progress_bar.value = len(configs_to_test)
            
            if not stop_attack:
                log("\n<span style='color: #2ecc71; font-weight: bold;'>==== Campaign Finished Naturally ====</span>")
            log("Connection closed. Ready for new parameters.\n")
            
            start_button.disabled = False

    # ==========================================
    # Start/Stop Button Logic
    # ==========================================

    def on_start_clicked(b):
        global stop_attack
        stop_attack = False
        start_button.disabled = True
        log_history.clear()
        output_log.value = ""
        
        reps = repetitions_input.value
        if reps <= 0: reps = 1

        # LOGIC 1: TARGETED MODE (Prioritized if checked)
        if enable_targeted_checkbox.value:
            try:
                raw_input = targeted_input.value.strip()
                if not raw_input:
                    raise ValueError("Textbox is empty. Paste a list like [(200, 10)]")
                
                parsed_list = ast.literal_eval(raw_input)
                
                if not isinstance(parsed_list, list):
                    raise ValueError("Input must be a valid Python list of tuples.")
                
                # Apply repetitions to the targeted list
                configs_to_test = [config for config in parsed_list for _ in range(reps)]
                log(f"Starting Targeted Mode: Testing {len(parsed_list)} configurations.")
                
            except Exception as e:
                log(f"<span style='color: #e74c3c;'><b>Targeted Input Error:</b> {e}</span>")
                log("<span style='color: #e74c3c;'>Attack aborted. Please check the Configs text box.</span>")
                start_button.disabled = False
                return

        # LOGIC 2: DELAY SWEEP MODE
        elif enable_scan_checkbox.value:
            start_val = start_delay_scan.value
            stop_val = stop_delay_scan.value
            step_val = step_delay_scan.value
            
            if step_val <= 0: step_val = 1 
            if stop_val < start_val: stop_val = start_val
                
            base_delays = list(range(start_val, stop_val + 1, step_val))
            
            # Attach the CURRENT UI PULSE to each delay step
            base_configs = [(d, pulse_slider.value) for d in base_delays]
            
            configs_to_test = [config for config in base_configs for _ in range(reps)]
            log(f"Starting Scan Mode: Sweeping Delay from {start_val} to {stop_val} by {step_val}.")

        # LOGIC 3: MANUAL MODE
        else:
            configs_to_test = [None] * n_tests_input.value
            log(f"Starting Manual Mode for {n_tests_input.value} tests.")

        log(f"-> Repetitions per parameter: {reps}")
        log(f"-> Total attacks queued: {len(configs_to_test)}")

        # Sync the progress bar
        progress_bar.max = len(configs_to_test)
        progress_bar.value = 0
        
        thread = threading.Thread(target=attack_task, args=(configs_to_test,))
        thread.start()

    def on_stop_clicked(b):
        global stop_attack
        stop_attack = True
        log("Stop requested... waiting for current cycle to finish.")

    start_button.on_click(on_start_clicked)
    stop_button.on_click(on_stop_clicked)

In [ ]:
run_attack_ecdsa_ui()
pico.disarm()